# GEP Navigator — pipeline walkthrough

This notebook mirrors the TypeScript pipeline stage by stage, reading the artefacts it writes. It needs only the Python standard library.

**PDF → runs with fonts → repair → outline → chunks (parent/child) → embeddings → index → route → hybrid search → rerank → parents → gate → generate → verify**

Run `npm run ingest` first (no API key needed). The last section calls the running app (`npm run dev`) for a live answer.

In [ ]:
import json, os, textwrap, urllib.request
from collections import Counter
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
idx = json.load(open(os.path.join(ROOT, 'data/index/index.json')))
print(f"{idx['pageCount']} pages · {len(idx['sections'])} sections · {len(idx['parents'])} parents · {len(idx['chunks'])} chunks · {len(idx['naive'])} naive chunks · {len(idx['cells'])} table cells")

## 1. Parsing: why not plain text

The naive baseline indexes raw page text in content-stream order: running heads spelled with spaces, chart axis labels, table cells out of order, sentences from two columns. (With pdftotext, ligatures also arrive as single codepoints, `ﬁ`; pdfjs decodes them, and NFKC normalisation covers either.) The structured pipeline reads each run's **font** (`src/lib/pdf/extract.ts`) and uses the report's typography to recover structure (`src/lib/pdf/structure.ts`).

In [ ]:
naive = next(c for c in idx['naive'] if 'G L O B A L' in c['text'] and 'Percent' in c['text'])
print('NAIVE CHUNK (PDF page', naive['pdfStart'], ')\n', naive['text'][:500])
print('\nnaive chunks carrying a spaced-out running head:', sum('G L O B A L' in c['text'] for c in idx['naive']), 'of', len(idx['naive']))
print('structured chunks carrying one:', sum('G L O B A L' in c['text'] for c in idx['chunks']))
# pdfjs decodes ligatures itself; pdftotext on the same PDF emits ~1,500 ligature codepoints (ﬁ ﬂ ﬀ ﬃ), which NFKC repairs.

## 2. The outline

Headings (Helvetica Bold 16 / Roman 11), boxes (Garamond 10.6), figures and tables (captions in Helvetica Bold 10) become a section tree. Every section carries chapter, region, box and kind.

In [ ]:
for s in idx['sections']:
    if s['level'] <= 1 and s['chapter'] in ('ES','1','2','3','4','SA') and s['kind'] != 'references':
        tag = ' '.join(x for x in [s['kind'], s['region'] or '', ('box '+s['boxId']) if s['boxId'] else ''] if x)
        print('  ' * s['level'] + s['title'][:70], f"[{tag}] p.{idx['printed'][s['pdfStart']-1]}")

## 3. Chunking

Parents (~900 tokens) are what the model reads; children (~280 tokens, 40 overlap) are what search sees. Neither crosses a section boundary. The heading path is embedded with each child; the stored text stays verbatim for citation checks.

In [ ]:
sar = next(c for c in idx['chunks'] if 'Growth in SAR is expected to soften to 6.3' in c['text'])
print('heading path :', sar['headingPath'])
print('region/kind  :', sar['region'], sar['kind'], '· printed p.', sar['printedStart'])
print('retrieval text (first 300):\n', sar['retrievalText'][:300])
print('\nchunks by kind:', dict(Counter(c['kind'] for c in idx['chunks'])))
print('chunks by region:', dict(Counter(c['region'] for c in idx['chunks'] if c['region'])))

## 4. Tables, rebuilt

Columns come from the header row's x positions; revision columns (differences from January 2026 projections) are marked; fiscal-year groups keep their labels. Every numeric cell lands in `cells` for the computed source.

In [ ]:
rows = [c for c in idx['cells'] if c['tableId'] == '2.10' and c['rowLabel'] in ('India', 'Maldives')]
for c in rows: print(f"{c['rowLabel']:<10} {c['column']:<10} {'revision' if c['isRevision'] else 'level':<9} {c['raw']}")
print('\ncountry → region (from the report\'s own tables):', {k: idx['countries'][k] for k in ('india','pakistan','brazil','mongolia','nigeria')})

## 5–8. Routing, retrieval, rerank, gate

These run in TypeScript (`npm run eval:retrieval`). The ladder adds one technique per rung, scored on whether the retrieved passage sits in the right chapter/region/pages.

In [ ]:
try:
    r = json.load(open(os.path.join(ROOT, 'data/eval/retrieval-latest.json')))
    print(f"{'rung':<44} hit@1  hit@5   MRR  wrong-region")
    for x in r['rungs']:
        print(f"{x['label']:<44} {x['hit1']:>5.0%} {x['hit5']:>6.0%} {x['mrr']:>5.2f} {x['mixup']:>8.0%}")
    if r.get('gate'): print(f"\nrefusal gate: threshold {r['gate']['threshold']} → accuracy {r['gate']['accuracy']:.0%}")
except FileNotFoundError:
    print('Run: npm run eval:retrieval')

## 9. Generate and verify (live)

With `npm run dev` running and a Gemini key set, ask the app. Citations come back already verified: each quote was found verbatim in its passage, and passages from outside the routed scope were refused.

In [ ]:
BASE = os.environ.get('GEP_BASE', 'http://localhost:3200')
def ask(q, mode='full'):
    req = urllib.request.Request(f'{BASE}/api/ask', data=json.dumps({'question': q, 'mode': mode}).encode(), headers={'content-type': 'application/json'})
    return json.load(urllib.request.urlopen(req, timeout=180))
try:
    res = ask('What is the growth outlook for South Asia in this report?')
    print('scope:', res['route']['regions'], res['route']['chapter'], '· refused:', res['refused'])
    print(textwrap.fill(res['answer'], 100))
    for c in res['citations']: print(f"  [{c['n']}] {c['label']}\n      “{c['span']}”")
except Exception as e:
    print('App not reachable or no key:', e)